In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from core.tensor import Tensor
from lattice.coder import PathCoder

In [ ]:
tensor = Tensor()
legs=[
        lambda x, rel, t: tensor.Join(x.reshape(1, -1), rel.T, t),
        lambda x, rel, t: tensor.Join(x.reshape(1, -1), rel, t).squeeze(),
        lambda x, rel, t: tensor.Residuate(rel, x.reshape(-1, 1), t).reshape(-1, 1),
        lambda x, rel, t: tensor.Residuate(rel.T, x.reshape(-1, 1), t).reshape(-1),
    ]

In [ ]:
if __name__ == "__main__":
    import numpy as np

    tensor = Tensor()

    emb = np.array([
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
    ], dtype=float)

    q_concept = np.array([1.0, 0.0], dtype=float)
    x_entity = np.array([1.0, 1.0, 0.0], dtype=float)
    temp = 0.0

    bicoder = PathCoder(legs, emb=emb)

    print("parse sepe:", bicoder.parse("sepe"))
    print("parse pese:", bicoder.parse("pese"))

    print("concept roundtrip sepe:", bicoder.run("sepe", q_concept, temp))
    print("entity roundtrip pese:", bicoder.run("pese", x_entity, temp))

    f = bicoder.op("sepe")
    print("callable sepe:", f(q_concept, temp))

In [ ]:
for step, name, shape, value in bicoder.trace("sepe", q_concept, temp):
    print(step, name, shape)
    print(value)

In [ ]:
import numpy as np

emb = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 0.0],
], dtype=float)

In [ ]:
emb = np.array([
    [1.0, 0.2, 0.0, 0.0],
    [0.8, 0.9, 0.1, 0.0],
    [0.1, 1.0, 0.7, 0.0],
    [0.0, 0.2, 1.0, 0.8],
    [0.0, 0.0, 0.3, 1.0],
    [0.7, 0.1, 0.9, 0.2],
], dtype=float)

In [ ]:
q1 = np.array([1.0, 0.0, 0.0, 0.0], dtype=float)
q2 = np.array([0.0, 1.0, 1.0, 0.0], dtype=float)
x1 = np.array([1.0, 1.0, 0.0, 0.0, 0.0, 1.0], dtype=float)

temp = 0.0

In [ ]:
tensor = Tensor()
bicoder = PathCoder(legs, emb=emb)

print(bicoder.explain("sepe"))
print()
print(bicoder.explain("pese"))

print("\n--- sepe on q1 ---")
for step, info, shape, value in bicoder.trace("sepe", q1, temp):
    print(step, info, shape)
    print(value)

print("\n--- sepe on q2 ---")
for step, info, shape, value in bicoder.trace("sepe", q2, temp):
    print(step, info, shape)
    print(value)

print("\n--- pese on x1 ---")
for step, info, shape, value in bicoder.trace("pese", x1, temp):
    print(step, info, shape)
    print(value)

In [1]:
!pip install -q torch torch-semiring-einsum --break-system-packages

In [2]:
import math
from lattice.coder import PathCoder
import torch
import torch_semiring_einsum as tse

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

print("torch:", torch.__version__)
print("module:", tse.__name__)

torch: 2.10.0+cu128
module: torch_semiring_einsum


In [3]:
# Cell 5: test relation and vectors
rel = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 1.0],
    [0.0, 0.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 0.0],
], dtype=torch.float32)

q1 = torch.tensor([1.0, 0.0, 0.0, 0.0], dtype=torch.float32)  # k=4
q2 = torch.tensor([0.0, 1.0, 1.0, 0.0], dtype=torch.float32)  # k=4
x1 = torch.tensor([1.0, 1.0, 0.0, 0.0, 0.0, 1.0], dtype=torch.float32)  # n=6

print("rel:", rel.shape)
print("q1 :", q1.shape)
print("x1 :", x1.shape)

rel: torch.Size([6, 4])
q1 : torch.Size([4])
x1 : torch.Size([6])


In [4]:
legs = [
    lambda x, rel, t: torch.einsum("k,nk->n", x, rel), 
    lambda x, rel, t: torch.einsum("n,nk->k", x, rel), 
    lambda x, rel, t: torch.einsum("n,nk->k", x, rel), 
    lambda x, rel, t: torch.einsum("k,nk->n", x, rel)
]

In [5]:
coder = PathCoder(legs, rel)

In [7]:
path_sepe = coder.op("sepe")
path_pese = coder.op("pese")
path_sesd = coder.op("sesd")
path_pepd = coder.op("pepd")

In [9]:
print("sepe(q1):", coder.run("sepe", q1, temp=0.0))
print("sepe(q2):", coder.run("sepe", q2, temp=0.0))
print("pese(x1):", coder.run("pese", x1, temp=0.0))

sepe(q1): tensor([3., 1., 1., 0.])
sepe(q2): tensor([2., 3., 4., 1.])
pese(x1): tensor([3., 4., 2., 1., 0., 4.])


In [11]:
print("sepe(q1):", path_sepe(q1, temp=0.0))
print("sepe(q2):", path_sepe(q2, temp=0.0))
print("pese(x1):", path_pese(x1, temp=0.0))

print("sesd(q1):", path_sesd(q1, temp=0.0))
print("pepd(x1):", path_pepd(x1, temp=0.0))

sepe(q1): tensor([3., 1., 1., 0.])
sepe(q2): tensor([2., 3., 4., 1.])
pese(x1): tensor([3., 4., 2., 1., 0., 4.])
sesd(q1): tensor([3., 1., 1., 0.])
pepd(x1): tensor([3., 4., 2., 1., 0., 4.])
